In [1]:
import os
import glob
import pandas as pd

In [2]:
english_objects = {"Fur ball", "Sponge ball", "BaseBall", "Steal ball", "bouncy ball"}
number_objects = {"0", "10", "30", "50", "100"}
valid_objects = english_objects.union(number_objects)

In [3]:
def process_single_file(file_path):
    df = pd.read_csv(file_path)
    df["ObjectName"] = df["ObjectName"].astype(str)
    df = df[df["ObjectName"].isin(valid_objects)].copy()

    df_english = df[df["ObjectName"].isin(english_objects)].copy()
    df_number = df[df["ObjectName"].isin(number_objects)].copy()

    english_valid = len(df_english) == 5
    number_valid = len(df_number) == 5

    return df_english if english_valid else None, df_number if number_valid else None, english_valid or number_valid


In [4]:
def process_folder(folder_path):
    all_data, english_data, number_data = [], [], []
    invalid_files = []

    file_list = sorted(glob.glob(os.path.join(folder_path, "*.csv")))  # 파일 순서 보장

    for session_id, file_path in enumerate(file_list, start=1):
        df_english, df_number, is_valid = process_single_file(file_path)
        file_name = os.path.basename(file_path)

        if df_english is not None:
            df_english["SourceFile"] = file_name
            df_english["session"] = session_id
            english_data.append(df_english)
            all_data.append(df_english)

        if df_number is not None:
            df_number["SourceFile"] = file_name
            df_number["session"] = session_id
            number_data.append(df_number)
            all_data.append(df_number)

        if not is_valid:
            invalid_files.append(file_name)

    # 정렬 및 열 이름 통일
    for df_list in (english_data, number_data, all_data):
        for d in df_list:
            d.rename(columns={"Zone": "zone"}, inplace=True)

    df_all = pd.concat(all_data, ignore_index=True).sort_values(["session", "Timestamp"])
    df_english = pd.concat(english_data, ignore_index=True).sort_values(["session", "Timestamp"])
    df_number = pd.concat(number_data, ignore_index=True).sort_values(["session", "Timestamp"])

    return df_all, df_english, df_number, invalid_files

In [5]:
def main(base_folder):
    print(base_folder)
    result = {}

    for name_folder in os.listdir(base_folder):
        path = os.path.join(base_folder, name_folder)
        if os.path.isdir(path):
            df_all, df_eng, df_num, invalids = process_folder(path)
            result[name_folder] = {
                "full": df_all,
                "english": df_eng,
                "number": df_num,
                "invalid_files": invalids
            }

    return result

In [6]:
results = main("./Sequence")

# 예시 출력
for name, data in results.items():
    print(f"\n=== {name} ===")
    print("📁 유효하지 않은 파일들:", data["invalid_files"])
    print("📊 전체 데이터:", data["full"].shape)
    print("📊 영문 데이터:", data["english"].shape)
    print("📊 숫자 데이터:", data["number"].shape)

./Sequence

=== bosung ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== dohoon ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== euncha ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== hyunwook ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== jaeho ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== jaehu ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== jinu ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== jiyoung ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== minho ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== minju ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (50, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (25, 5)

=== sarang ===
📁 유효하지 않은 파일들: []
📊 전체 데이터: (55, 5)
📊 영문 데이터: (25, 5)
📊 숫자 데이터: (30, 5)

=== seongjun ===
📁 유효하지

In [7]:
output_dir = ".\Full\Sequence_Full\Full_data"

In [8]:
for name, data in results.items():
    print(f"\n=== {name} ===")
    # print("📊 전체 데이터:", data["full"])
    # print("📊 영문 데이터:", data["english"])
    # print("📊 숫자 데이터:", data["number"])
    
    # === CSV로 저장 ===
    export_path = os.path.join(output_dir, f"{name}_full.csv")
    data["full"].to_csv(export_path, index=False, encoding='utf-8-sig')  # utf-8-sig로 한글 깨짐 방지
    print(f"✅ {export_path} 저장 완료!")


=== bosung ===
✅ .\Full\Sequence_Full\Full_data\bosung_full.csv 저장 완료!

=== dohoon ===
✅ .\Full\Sequence_Full\Full_data\dohoon_full.csv 저장 완료!

=== euncha ===
✅ .\Full\Sequence_Full\Full_data\euncha_full.csv 저장 완료!

=== hyunwook ===
✅ .\Full\Sequence_Full\Full_data\hyunwook_full.csv 저장 완료!

=== jaeho ===
✅ .\Full\Sequence_Full\Full_data\jaeho_full.csv 저장 완료!

=== jaehu ===
✅ .\Full\Sequence_Full\Full_data\jaehu_full.csv 저장 완료!

=== jinu ===
✅ .\Full\Sequence_Full\Full_data\jinu_full.csv 저장 완료!

=== jiyoung ===
✅ .\Full\Sequence_Full\Full_data\jiyoung_full.csv 저장 완료!

=== minho ===
✅ .\Full\Sequence_Full\Full_data\minho_full.csv 저장 완료!

=== minju ===
✅ .\Full\Sequence_Full\Full_data\minju_full.csv 저장 완료!

=== sarang ===
✅ .\Full\Sequence_Full\Full_data\sarang_full.csv 저장 완료!

=== seongjun ===
✅ .\Full\Sequence_Full\Full_data\seongjun_full.csv 저장 완료!

=== siu ===
✅ .\Full\Sequence_Full\Full_data\siu_full.csv 저장 완료!

=== suhan ===
✅ .\Full\Sequence_Full\Full_data\suhan_full.csv 저장 완료!

=